<a href="https://colab.research.google.com/github/Shorovpaul/Data_Testing/blob/main/Diabetes_012_health_indicators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas numpy scikit-learn imbalanced-learn shap xgboost

In [3]:
import pandas as pd

data = pd.read_csv("diabetes_012_health_indicators.csv")

print(data.shape)
# Step 1: Rename the column
data.rename(columns={'Diabetes_012': 'Class'}, inplace=True)

# Step 2: Verify it worked
print(data.columns.tolist())

# Step 3: Now use it
print(data['Class'])
print(data['Class'].value_counts())

(245670, 22)
['Class', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']
0         0.0
1         0.0
2         0.0
3         0.0
4         0.0
         ... 
245665    0.0
245666    0.0
245667    0.0
245668    0.0
245669    0.0
Name: Class, Length: 245670, dtype: float64
Class
0.0    207222
2.0     34006
1.0      4442
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

X = data.drop("Class", axis=1)
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [7]:
print('Missing values in X_train before imputation:')
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

# Impute missing values with the median of each column
for col in X_train.columns:
    if X_train[col].isnull().any():
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val) # Modified line to address FutureWarning

print('\nMissing values in X_train after imputation:')
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('\nValue counts after SMOTE:')
print(y_train_bal.value_counts())

Missing values in X_train before imputation:
Series([], dtype: int64)

Missing values in X_train after imputation:
Series([], dtype: int64)

Value counts after SMOTE:
Class
0.0    165778
2.0    165778
1.0    165778
Name: count, dtype: int64


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
import xgboost as xgb

models = {
    "LR": LogisticRegression(max_iter=1000),
    "RF": RandomForestClassifier(n_estimators=200),
    "GB": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True),
    "XGB": xgb.XGBClassifier(eval_metric="logloss")
}

trained_models = {}

for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    trained_models[name] = model

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
